# 五会话局部价值的日间与日内方差审计

冻结`plan.json`，只读已验证Qlib标签Recorder。此卡诊断目标/时钟匹配；不拟合、不选时间、不读取2021/22经济结果。

In [1]:
import os,json,hashlib
from pathlib import Path
os.environ['MLFLOW_ALLOW_FILE_STORE']='true';os.environ['MLFLOW_DISABLE_AGENT_HINT']='1'
import numpy as np,pandas as pd,qlib
from qlib.data import D
from src.core.recorder_sources import load_verified_recorder_objects
root=Path.cwd();card=root/'research/stage1_v5_label_clock_variance_audit_20261030';plan=json.loads((card/'plan.json').read_text())
sha=lambda p:hashlib.sha256(Path(p).read_bytes()).hexdigest()
src=root/plan['source_card'];assert sha(src/'summary.json')==plan['source_summary_sha256']
assert sha(src/'verification_manifest.json')==plan['source_manifest_sha256']
assert json.loads((src/'nas_binding.json').read_text())['evidence']['sha256']==plan['source_nas_archive_sha256']
assert sha(root/plan['provider_uri']/'qlib_store_manifest.json')==plan['provider_manifest_sha256']
qlib.init(provider_uri=str(root/plan['provider_uri']),region='us',kernels=1)
cal=pd.DatetimeIndex(D.calendar(start_time='2018-10-15 09:31:00',end_time='2020-12-31 16:00:00',freq='1min'))
objects,_=load_verified_recorder_objects(src,expected_recorder_id=plan['source_recorder_id'],artifacts=['analysis/labels'],expected_manifest_sha256=plan['source_manifest_sha256'],expected_artifact_sha256={'analysis/labels':plan['source_artifact_sha256']})
full=objects['analysis/labels'];assert len(full)==385932
sample=full.loc[full.trade_date.dt.year.isin(plan['discovery_years'])].copy()
assert sample.trade_date.dt.year.max()==2020
assert pd.DatetimeIndex(sample.bar_end_at.unique()).isin(cal).all()
unknown=sample.loc[~sample.status.isin(['paired_funded','attempt_no_fill'])]
x=sample.loc[sample.status.isin(['paired_funded','attempt_no_fill'])].copy()
assert x[plan['labels']].notna().all().all() and x.duplicated(['trade_date','bar_end_at']).sum()==0
x=x.sort_values(['trade_date','bar_end_at']).reset_index(drop=True)
def metrics(part,label):
    grouped=part.groupby('trade_date',sort=True)[label]
    means=grouped.mean();grand=float(means.mean())
    within=float(grouped.apply(lambda v:np.mean((v-v.mean())**2)).mean())
    between=float(np.mean((means.to_numpy(float)-grand)**2))
    total=float(within+between)
    # independent formula identity: equal-day mean of each minute squared deviation
    direct=float(grouped.apply(lambda v:np.mean((v-grand)**2)).mean())
    assert abs(direct-total)<=max(1e-15,abs(total)*1e-10)
    ranges=grouped.max()-grouped.min()
    first=grouped.first();last=grouped.last()
    return {'days':int(len(means)),'rows':int(len(part)),'grand_mean_native':grand,'grand_mean_bp':grand*1e4 if label in ('soxx_value','soxl_value','relative_value_soxl_minus_soxx') else None,
            'within_variance':within,'between_variance':between,'total_variance':total,
            'within_fraction':within/total if total>0 else None,
            'daily_range_p10_bp':float(ranges.quantile(.1)*1e4) if label in ('soxx_value','soxl_value','relative_value_soxl_minus_soxx') else None,
            'daily_range_p50_bp':float(ranges.quantile(.5)*1e4) if label in ('soxx_value','soxl_value','relative_value_soxl_minus_soxx') else None,
            'daily_range_p90_bp':float(ranges.quantile(.9)*1e4) if label in ('soxx_value','soxl_value','relative_value_soxl_minus_soxx') else None,
            'first_minus_last_mean_bp':float((first-last).mean()*1e4) if label in ('soxx_value','soxl_value','relative_value_soxl_minus_soxx') else None}
all_result={label:metrics(x,label) for label in plan['labels']}
yearly={str(year):{label:metrics(x.loc[x.trade_date.dt.year.eq(year)],label) for label in plan['labels']} for year in plan['discovery_years']}
quarters={str(q):{label:metrics(g,label) for label in plan['labels']} for q,g in x.groupby(x.trade_date.dt.to_period('Q'),sort=True)}
counts=x.groupby('trade_date').size()
result={'experiment':plan['experiment'],'plan_sha256':sha(card/'plan.json'),'source_recorder_id':plan['source_recorder_id'],
 'source_rows':int(len(sample)),'mature_rows':int(len(x)),'unknown_rows':int(len(unknown)),
 'source_days':int(sample.trade_date.nunique()),'mature_days':int(x.trade_date.nunique()),
 'minute_count_per_day_min':int(counts.min()),'minute_count_per_day_median':float(counts.median()),'minute_count_per_day_max':int(counts.max()),
 'all_discovery':all_result,'annual':yearly,'quarterly':quarters,
 'new_model_fits':0,'new_accounts':0,'new_recorders':0,'historical_research_passed':False,
 'decision':'label_clock_geometry_diagnostic_no_candidate_policy'}
(card/'audit.json').write_text(json.dumps(result,ensure_ascii=False,indent=2,allow_nan=False)+'\n')
print(json.dumps({'source_rows':result['source_rows'],'mature_rows':result['mature_rows'],'mature_days':result['mature_days'],'all_discovery':all_result,'annual':{y:{k:v['within_fraction'] for k,v in d.items()} for y,d in yearly.items()}},ensure_ascii=False,indent=2,allow_nan=False))


[61718:MainThread](2026-10-30 18:43:12,034) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


[61718:MainThread](2026-10-30 18:43:12,435) INFO - qlib.Initialization - [__init__.py:75] - qlib successfully initialized based on client settings.


[61718:MainThread](2026-10-30 18:43:12,435) INFO - qlib.Initialization - [__init__.py:77] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/dev/workspace/data/qlib/stage1_event_guard_fullminute_20261028')}


{
  "source_rows": 203526,
  "mature_rows": 202058,
  "mature_days": 554,
  "all_discovery": {
    "soxx_value": {
      "days": 554,
      "rows": 202058,
      "grand_mean_native": 0.0008961576855072158,
      "grand_mean_bp": 8.961576855072158,
      "within_variance": 2.8130707144912417e-07,
      "between_variance": 1.6450660022913395e-05,
      "total_variance": 1.673196709436252e-05,
      "within_fraction": 0.016812552275691756,
      "daily_range_p10_bp": 8.027495333878038,
      "daily_range_p50_bp": 14.578218780443667,
      "daily_range_p90_bp": 32.92961752779402,
      "first_minus_last_mean_bp": 0.21689956400716248
    },
    "soxl_value": {
      "days": 554,
      "rows": 202058,
      "grand_mean_native": 0.0008153701673778389,
      "grand_mean_bp": 8.15370167377839,
      "within_variance": 3.195054667284208e-07,
      "between_variance": 1.6385150643304542e-05,
      "total_variance": 1.6704656110032964e-05,
      "within_fraction": 0.019126731171467997,
      "dail

# 五会话局部价值的日间与日内时钟审计

**SOXX 判定：历史 Goal 未通过。**只读已获 NAS 收据/限定读回的固定原价股数五会话标签 Qlib Recorder `b85e80ab42fa4d9c83c6c9667114fd5d`；没有新模型、候选完整自身账户或0/2/5bp SOXX门。2018–2020共有203526个来源分钟意向，其中202058条标签成熟、覆盖554日，1468条未知原样保留；2021/22经济值未访问。每个交易日总权重为1，按日均值把费用后局部值分解为日间与日内方差，并逐年/逐季度保存于`audit.json`。

SOXX相对HOLD的五会话局部价值总方差中，**日内份额仅1.681%**，SOXL相对HOLD为**1.913%**；SOXL−SOXX资产选择差为**11.962%**。SOXX日内份额在部分2018Q4、完整2019、完整2020分别2.571%/1.205%/1.820%；资产差对应9.797%/10.945%/12.434%。每日SOXX局部值的分钟范围中位14.578bp，范围非零并不等于可以在事前挑中最好分钟。全部分钟高度依赖；方差份额也**不是分钟特征预测力的数学上界**，分钟信息仍可能预示日间结果。

弱/强区间对照：2020Q1的SOXX局部日均−8.730bp、日内份额1.932%，2020Q2为+24.434bp、2.119%；2018Q4局部均值−7.249bp，2019Q1 +19.320bp。已观察差别主要落在交易日之间，与旧高维五会话模型按分钟点损失拟合、2022Q2首分钟全部买入及保存模型有效样本仅549训练日的失败链路一致。但本卡**未识别**是价格路径、现金机会损失、费用、SOXL选择还是风险反馈的哪一项造成收益；季度只是事后归因，不能制定切换阈值。

方法修正：不再把202058个发现期分钟标签当成独立的五会话方向训练样本，也不通过第一根bar收盘位置换窗口拟合（同类close-location字段已在旧因子/搜索代码中使用）。下一机制若主张分钟择时，应先有明确的短时可执行经济目标和不同决策时信息，按日及五会话重叠隔离；若主张五会话资产/现金选择，应先解决日级有效样本与模型校准，再证明每分钟分数能改变合法整股行动。两者最后都须经过连续三资产、同风险域、自身全账户的SOXX合同门。本卡仅改变研究目标诊断，不能授权重训或宣称alpha。

首次计算把相对价值和尾部损伤的展示单位误标；`plan_v1_before_unit_fix.json`、`audit_v1_before_unit_fix.json`和`attempt1_unit_contract_failure.json`保留。第二次只修正单位映射，不改来源、公式、年份或结果。